In [ ]:
# =============================================================================
# MAKERSPACE WASTE ANALYSIS - JACKSON INNOVATION STUDIO, UIUC
# =============================================================================
# This script calculates all key metrics from 4 data collection events:
#   - Waste Rate (g/day)
#   - Material Usage Rate (g/day)
#   - Scrap Ratio (%)
#
# Data collection dates:
#   Event 1 (E1): October 13, 2025
#   Event 2 (E2): December 16, 2025   ← end of Fall 2025 semester
#   Event 3 (E3): February 13, 2026   ← start of Spring 2026 semester
#   Event 4 (E4): March 30, 2026
# =============================================================================

from datetime import date

# =============================================================================
# SECTION 1: UNIT CONVERSION CONSTANTS
# =============================================================================

LBS_TO_G  = 453.592   # 1 pound = 453.592 grams
G_PER_ROLL = 1000.0   # Each filament spool is 1 kg = 1000 g (standard Ultimaker S5 spool)


# =============================================================================
# SECTION 2: WASTE DATA
# =============================================================================
# At each event, all waste in the studio was collected and weighed.
# For E1 and E2, the volume was large so only a SAMPLE was categorized by
# failure type. For E3 and E4, the full batch was categorized.
#
# IMPORTANT: Waste collected AT an event represents waste accumulated SINCE
# the previous event. So:
#   - Waste at E1 = accumulated before the study started (no defined start date)
#   - Waste at E2 = accumulated during E1 → E2 period  → used for E1→E2 rate
#   - Waste at E3 = accumulated during E2 → E3 period  → used for E2→E3 rate
#   - Waste at E4 = accumulated during E3 → E4 period  → used for E3→E4 rate

# --- Total waste weights (entire batch, in lbs) ---
# E1: 132 lbs total, but only 17.16 lbs (13%) was sampled for categorization
# E2: 58 lbs total, only 9.5 lbs sampled
# E3: 7.60 lbs total — fully categorized (small enough to sort entirely)
# E4: 3.63 lbs total — fully categorized

total_waste_lbs = {
    "E1": 132.0,          # pre-study baseline waste
    "E2": 58.0,           # waste from E1→E2 period
    "E3": 7.59712052,     # waste from E2→E3 period
    "E4": 3.6265999,      # waste from E3→E4 period
}

# Convert all waste to grams
total_waste_g = {event: lbs*LBS_TO_G for event, lbs in total_waste_lbs.items()}


# =============================================================================
# SECTION 3: TIME PERIODS
# =============================================================================
# Calculate the number of days between each consecutive pair of events.
# These are used as the denominators for all rate calculations.

event_dates = {
    "E1": date(2024, 10, 13),
    "E2": date(2024, 12, 16),
    "E3": date(2025,  2, 13),
    "E4": date(2025,  3, 30),
}

days_E1_E2 = (event_dates["E2"] - event_dates["E1"]).days  # 64 days
days_E2_E3 = (event_dates["E3"] - event_dates["E2"]).days  # 59 days
days_E3_E4 = (event_dates["E4"] - event_dates["E3"]).days  # 45 days


# =============================================================================
# SECTION 4: WASTE RATE (g/day)
# =============================================================================
# Formula: Waste Rate = Waste collected at END of period / Days in period
#
# Example for E1→E2:
#   Waste collected at E2 = 58 lbs = 26,308.34 g
#   Days in period        = 64
#   Waste rate            = 26,308.34 / 64 = 411.07 g/day
#
# Note: E1 waste (132 lbs) cannot be assigned a rate because we don't
# know when the accumulation period started.

waste_rate_E1_E2 = total_waste_g["E2"] / days_E1_E2   # g/day
waste_rate_E2_E3 = total_waste_g["E3"] / days_E2_E3   # g/day
waste_rate_E3_E4 = total_waste_g["E4"] / days_E3_E4   # g/day

print("\n")
print("=" * 65)
print("SECTION 4: WASTE RATE (g/day)")
print("=" * 65)
print(f"""
  Formula: Waste Rate = Waste collected at end of period / Days in period

  E1→E2:
    Waste at E2     = {total_waste_lbs['E2']} lbs × {LBS_TO_G} g/lb = {total_waste_g['E2']:,.2f} g
    Days            = {days_E1_E2}
    Waste Rate      = {total_waste_g['E2']:,.2f} / {days_E1_E2} = {waste_rate_E1_E2:.2f} g/day

  E2→E3:
    Waste at E3     = {total_waste_lbs['E3']:.5f} lbs × {LBS_TO_G} g/lb = {total_waste_g['E3']:,.2f} g
    Days            = {days_E2_E3}
    Waste Rate      = {total_waste_g['E3']:,.2f} / {days_E2_E3} = {waste_rate_E2_E3:.2f} g/day

  E3→E4:
    Waste at E4     = {total_waste_lbs['E4']:.7f} lbs × {LBS_TO_G} g/lb = {total_waste_g['E4']:,.2f} g
    Days            = {days_E3_E4}
    Waste Rate      = {total_waste_g['E4']:,.2f} / {days_E3_E4} = {waste_rate_E3_E4:.2f} g/day
""")


# =============================================================================
# SECTION 5: FILAMENT INVENTORY
# =============================================================================
# At E2 and beyond, filament inventory was taken at each event.
# Inventory = full rolls on shelves + partial rolls on printers (estimated visually)
#
# E2 inventory:
#   Outside (shelves): 28.25 rolls
#   On printers:       23.00 rolls
#   Total:             51.25 rolls = 51,250 g
#
# Between E2 and E3 (semester break):
#   96 new rolls were ordered → adds 96,000 g to available filament
#
# E3 inventory:
#   Full rolls:        67.00 rolls
#   On printers:       16.25 rolls (each printer's spool visually estimated,
#                      e.g. 80% full → rounded to 0.75, etc.)
#   Total:             83.25 rolls = 83,250 g
#
# E4 inventory:
#   Full rolls:        23.00 rolls
#   On printers:       17.25 rolls
#   Total:             40.25 rolls = 40,250 g
#
# Mid-semester reorder detection (from studio's weekly inventory data):
#   The studio tracks rolls every week. The weekly data shows:
#     3/24 → 26 rolls, then 4/2 → 83 rolls  (a jump of +57 rolls)
#   Since E4 = 3/30, which is BEFORE 4/2, this reorder did NOT occur
#   during the E3→E4 period. So no new rolls are added for that period.

inventory_rolls = {
    "E2": 51.25,   # rolls at Event 2
    "E3": 83.25,   # rolls at Event 3
    "E4": 40.25,   # rolls at Event 4
}

new_rolls_between_E2_E3 = 96.0   # ordered during semester break
new_rolls_between_E3_E4 = 0.0    # reorder happened AFTER E4 date (3/30 < 4/2)

# Convert inventory to grams
inventory_g = {event: rolls * G_PER_ROLL for event, rolls in inventory_rolls.items()}


# =============================================================================
# SECTION 6: MATERIAL USAGE & USAGE RATE (g/day)
# =============================================================================
# Formula:
#   Material Used = Inventory at start + New rolls added - Inventory at end
#   Usage Rate    = Material Used / Days in period
#
# This can only be calculated for periods where we have inventory at BOTH ends,
# i.e., E2→E3 and E3→E4. We have no inventory at E1, so E1→E2 is unavailable.
#
# Example for E2→E3:
#   Start (E2):   51,250 g
#   + New rolls:  96,000 g  (ordered during semester break)
#   - End (E3):   83,250 g
#   = Used:       64,000 g
#   Usage Rate:   64,000 / 59 = 1,084.75 g/day

material_used_E2_E3 = (inventory_g["E2"]
                       + new_rolls_between_E2_E3 * G_PER_ROLL
                       - inventory_g["E3"])

material_used_E3_E4 = (inventory_g["E3"]
                       + new_rolls_between_E3_E4 * G_PER_ROLL
                       - inventory_g["E4"])

usage_rate_E2_E3 = material_used_E2_E3 / days_E2_E3   # g/day
usage_rate_E3_E4 = material_used_E3_E4 / days_E3_E4   # g/day

print("=" * 65)
print("SECTION 6: MATERIAL USAGE & USAGE RATE (g/day)")
print("=" * 65)
print(f"""
  Formula: Material Used = Start inventory + New rolls - End inventory
           Usage Rate    = Material Used / Days in period

  E2→E3:
    Start (E2):    {inventory_g['E2']:>10,.0f} g  ({inventory_rolls['E2']} rolls)
    + New rolls:   {new_rolls_between_E2_E3*G_PER_ROLL:>10,.0f} g  ({new_rolls_between_E2_E3:.0f} rolls ordered during semester break)
    - End (E3):    {inventory_g['E3']:>10,.0f} g  ({inventory_rolls['E3']} rolls)
    = Used:        {material_used_E2_E3:>10,.0f} g
    Days:          {days_E2_E3}
    Usage Rate:    {material_used_E2_E3:,.0f} / {days_E2_E3} = {usage_rate_E2_E3:.2f} g/day

  E3→E4:
    Start (E3):    {inventory_g['E3']:>10,.0f} g  ({inventory_rolls['E3']} rolls)
    + New rolls:   {new_rolls_between_E3_E4*G_PER_ROLL:>10,.0f} g  (reorder occurred after E4 date)
    - End (E4):    {inventory_g['E4']:>10,.0f} g  ({inventory_rolls['E4']} rolls)
    = Used:        {material_used_E3_E4:>10,.0f} g
    Days:          {days_E3_E4}
    Usage Rate:    {material_used_E3_E4:,.0f} / {days_E3_E4} = {usage_rate_E3_E4:.2f} g/day

  E1→E2: Cannot be calculated (no inventory taken at E1)
""")


# =============================================================================
# SECTION 7: SCRAP RATIO
# =============================================================================
# Formula: Scrap Ratio = Total Waste / Total Material Used  (expressed as %)
#
# The scrap ratio answers: "Of all the filament consumed, what fraction
# ended up as waste?"
#
# We use the waste collected at the END of each period (which represents
# waste accumulated DURING that period) against material used in that period.
#
# Example for E2→E3:
#   Waste during period   = waste at E3 = 3,446 g
#   Material used         = 64,000 g
#   Scrap Ratio           = 3,446 / 64,000 = 5.38%

scrap_ratio_E2_E3 = total_waste_g["E3"] / material_used_E2_E3
scrap_ratio_E3_E4 = total_waste_g["E4"] / material_used_E3_E4

# Overall: sum waste and usage across both periods
total_waste_both  = total_waste_g["E3"] + total_waste_g["E4"]
total_used_both   = material_used_E2_E3 + material_used_E3_E4
scrap_ratio_overall = total_waste_both / total_used_both

print("=" * 65)
print("SECTION 7: SCRAP RATIO")
print("=" * 65)
print(f"""
  Formula: Scrap Ratio = Waste during period / Material used during period

  E2→E3:
    Waste (E3 collection): {total_waste_g['E3']:,.2f} g
    Material used:         {material_used_E2_E3:,.0f} g
    Scrap Ratio:           {total_waste_g['E3']:,.2f} / {material_used_E2_E3:,.0f} = {scrap_ratio_E2_E3*100:.2f}%

  E3→E4:
    Waste (E4 collection): {total_waste_g['E4']:,.2f} g
    Material used:         {material_used_E3_E4:,.0f} g
    Scrap Ratio:           {total_waste_g['E4']:,.2f} / {material_used_E3_E4:,.0f} = {scrap_ratio_E3_E4*100:.2f}%

  Overall (E2→E4 combined):
    Total waste:           {total_waste_both:,.2f} g
    Total material used:   {total_used_both:,.0f} g
    Scrap Ratio:           {total_waste_both:,.2f} / {total_used_both:,.0f} = {scrap_ratio_overall*100:.2f}%

  Note: E1→E2 scrap ratio cannot be calculated (no inventory at E1).
""")


# =============================================================================
# SECTION 8: SUMMARY TABLE
# =============================================================================

print("=" * 65)
print("SECTION 8: FINAL SUMMARY TABLE")
print("=" * 65)

header = f"\n{'Period':<10} {'Waste (g)':<14} {'Days':<7} {'Waste Rate':>12} {'Used (g)':>12} {'Usage Rate':>12} {'Scrap Ratio':>12}"
divider = "-" * 81
print(header)
print(divider)

# E1→E2: no usage rate or scrap ratio
print(f"{'E1→E2':<10} {total_waste_g['E2']:<14,.2f} {days_E1_E2:<7} {waste_rate_E1_E2:>11.2f}  {'N/A':>12} {'N/A':>12} {'N/A':>12}")

# E2→E3
print(f"{'E2→E3':<10} {total_waste_g['E3']:<14,.2f} {days_E2_E3:<7} {waste_rate_E2_E3:>11.2f}  {material_used_E2_E3:>12,.0f} {usage_rate_E2_E3:>11.2f}  {scrap_ratio_E2_E3*100:>10.2f}%")

# E3→E4
print(f"{'E3→E4':<10} {total_waste_g['E4']:<14,.2f} {days_E3_E4:<7} {waste_rate_E3_E4:>11.2f}  {material_used_E3_E4:>12,.0f} {usage_rate_E3_E4:>11.2f}  {scrap_ratio_E3_E4*100:>10.2f}%")

print(divider)
print(f"{'Overall':<10} {total_waste_both:<14,.2f} {'—':<7} {'—':>12} {total_used_both:>12,.0f} {'—':>12} {scrap_ratio_overall*100:>10.2f}%")
print(f"\n  Units: Waste Rate and Usage Rate are in g/day")
print(f"         Scrap Ratio = Waste / Material Used × 100\n")